# 14장. 파인튜닝과 나만의 챗봇

사전학습한 이야기 GPT를 요청-이야기 데이터로 5분 동안 파인튜닝해, 요청을 따르는 이야기 챗봇을 만듭니다.

- **파인튜닝(실습 14-4)에는 GPU가 필요합니다.** 런타임 → 런타임 유형 변경 → T4 GPU를 고르세요.
- GPU를 받을 수 없다면 실습 14-4는 건너뛰세요. 실습 14-5부터는 저자가 파인튜닝한 모델을 자동으로 내려받아 씁니다.
- 드라이브에 13장에서 학습한 pretrain.pt가 없으면 저자의 사전학습 모델을 씁니다.

데이터: [Tiny-Ko-Stories SFT](https://huggingface.co/datasets/JosephLee/tiny-ko-stories-sft) (CC BY 4.0) · 모델: [JosephLee/modu-llm](https://huggingface.co/JosephLee/modu-llm) (CC BY 4.0)

## 실습 14-1 준비: 사전학습 모델 불러오기

In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm-v2
import torch
from modu_llm import Vocab, END, encode_texts, generate, format_pair
from modu_llm import load_checkpoint, load_sft_texts, get_file, train, mount_drive
from modu_llm import get_device

device = get_device(need_gpu=True)
vocab = Vocab.build()
folder = mount_drive()
pre_model, _, pre_step, _ = load_checkpoint(get_file("pretrain.pt"), "cpu")
print(f"사전학습 모델: {pre_step}걸음")

## 실습 14-2 사전학습 모델에 요청 넣어 보기

In [ ]:
question = format_pair("숲속에서 토끼 보리가 나오는 이야기를 들려줘")
print(repr(question))
torch.manual_seed(0)
answer = generate(pre_model, vocab, question, max_new_tokens=150)
print(question + answer)

## 실습 14-3 요청-이야기 데이터 살펴보기

In [ ]:
texts = load_sft_texts("train")
val_texts = load_sft_texts("validation")
print(f"학습용 {len(texts):,}쌍 | 검증용 {len(val_texts):,}쌍")
print(texts[0] + END)

## 실습 14-4 요청을 따르도록 파인튜닝하기 (GPU, 약 5분)

In [ ]:
train_data = encode_texts(texts, vocab)
val_data = encode_texts(val_texts, vocab)
sft_path = f"{folder}/sft.pt"

model, optimizer, _, _ = load_checkpoint(get_file("pretrain.pt"), device, optimizer_lr=1e-4)
step, history = train(model, optimizer, train_data, val_data,
                      minutes=5, ckpt_path=sft_path, eval_every=200,
                      keep_best=True)
print(f"학습 끝: {step}걸음")

## 실습 14-5 같은 요청, 달라진 대답

드라이브에 파인튜닝한 모델(sft.pt)이 있으면 그것을, 없으면 저자의 모델을 불러옵니다.

In [ ]:
model, _, best_step, _ = load_checkpoint(get_file("sft.pt"), "cpu")
print(f"파인튜닝 모델: {best_step}걸음")
for r in ["숲속에서 토끼 보리가 나오는 이야기를 들려줘", "강아지 뭉치가 나오는 이야기를 만들어 줘"]:
    print("요청:", r)
    for name, m in [("사전학습", pre_model), ("파인튜닝", model)]:
        torch.manual_seed(0)
        answer = generate(m, vocab, format_pair(r), max_new_tokens=120)
        print(f"  [{name}]", answer.replace("\n", " "))

## 실습 14-6 chat 함수 만들기

In [ ]:
def chat(request, temperature=0.8):
    question = format_pair(request)                        # ① 요청을 형식에 담기
    story = generate(model, vocab, question, max_new_tokens=300,
                     temperature=temperature, top_k=20)    # ② 이야기 이어 쓰기
    return story.strip()                                   # ③ 앞뒤 빈칸 정리하기

torch.manual_seed(1)
print(chat("강아지 뭉치가 부엌에서 겪은 이야기를 들려줘"))

## 실습 14-7 대화 화면 띄우기

화면에 나온 주소를 누르면 새 창에서도 열 수 있습니다. 요청을 바꾸고 Submit을 눌러 보세요.

In [ ]:
%pip install -q gradio
from modu_llm import chat_app

demo = chat_app(chat)
demo.launch()

## 실습 14-8 작은 모델의 한계 알기

In [ ]:
for r in ["대한민국의 수도는 어디야?", "Tell me a story about a cat."]:
    torch.manual_seed(0)
    print("요청:", r)
    print(chat(r)[:150])
    print()